In [104]:
import os
import sys
import pandas as pd

# `classifier` 모듈을 임포트하기 위해 프로젝트 루트를 경로에 추가합니다.
# 이 노트북이 `notebooks` 디렉토리 안에 있다고 가정합니다.
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

from notebooks.classifier import SigLIPClassifier

# SigLIP Batch Classification & CSV Export

이 노트북은 SigLIP 모델을 사용하여 배치 이미지 분류를 수행하고 결과를 CSV로 저장합니다.

## 1. 모델 설정

In [105]:
# 학습된 모델 가중치 경로 (파인튜닝된 모델 사용 시)

# SigLIP + Linear Head 모델
# CHECKPOINT_PATH = "../models/checkpoints/siglip-base-patch16-224/best_model_siglip-base-patch16-224_20251202_004235.pth"

# SigLIP + ML Decoder 모델
# CHECKPOINT_PATH = "../models/checkpoints/siglip-base-patch16-224-mldecoder/best_model_siglip-base-patch16-224-mldecoder_20260106_135605.pth"

# SigLIP2 + Linear Head 모델
# CHECKPOINT_PATH = "../models/checkpoints/siglip2-base-patch16-224/best_model_siglip2-base-patch16-224_20251202_024457.pth"

# SigLIP2 + ML Decoder 모델
CHECKPOINT_PATH = "../models/checkpoints/siglip2-base-patch16-224-mldecoder/best_model_siglip2-base-patch16-224-mldecoder_20260106_141348.pth"

# 학습 시 사용한 기본 모델 ID
# MODEL_ID = "google/siglip-base-patch16-224"
MODEL_ID = "google/siglip2-base-patch16-224"

# 분류 임계값 (Sigmoid threshold)
THRESHOLD = 0.75

# 분류기 초기화
# ML Decoder 체크포인트를 사용하면 자동으로 ML Decoder 모델이 로드됩니다
classifier = SigLIPClassifier(
    model_id=MODEL_ID,
    checkpoint_path=CHECKPOINT_PATH,
    threshold=THRESHOLD
)

모델 로딩 중...
파인튜닝된 가중치 로딩: ../models/checkpoints/siglip2-base-patch16-224-mldecoder/best_model_siglip2-base-patch16-224-mldecoder_20260106_141348.pth
체크포인트에서 ML Decoder 설정을 감지했습니다.
'google/siglip2-base-patch16-224' 모델을 로드합니다... (config: /tmp/tmp0w0qq9pc.yaml)
레이블 수: 5


Some weights of SiglipForImageClassification were not initialized from the model checkpoint at google/siglip2-base-patch16-224 and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.



=== ML Decoder 적용 ===
Classifier head replaced with ML Decoder
  Hidden size: 768
  Num queries: 5
  Num layers: 1
  Num heads: 8
  ML Decoder parameters: 7,887,109
freeze_vision_encoder=False: Vision Encoder를 동결하지 않습니다.
모델 가중치 로드 완료.
모델이 cuda 디바이스로 이동되었습니다.


## 2. 테스트 이미지 로드

In [106]:
# 테스트 이미지가 있는 폴더 경로
test_folder = '../../test_images/images/'

# 폴더 내 모든 이미지 파일 찾기
test_paths = []
for filename in os.listdir(test_folder):
    if filename.endswith(('.jpg', '.jpeg', '.png')):
        full_path = os.path.join(test_folder, filename)
        test_paths.append(full_path)

print(f"총 {len(test_paths)}개의 테스트 이미지를 찾았습니다.")

총 107개의 테스트 이미지를 찾았습니다.


## 3. 배치 분류 실행

In [107]:
# 배치 분류 수행
results = classifier.batch_classify(test_paths, threshold=THRESHOLD)

처리 중: 1/107 - 0046.jpg
처리 중: 2/107 - 0069.jpg
처리 중: 3/107 - 0052.jpg
처리 중: 4/107 - 0020.jpg
처리 중: 5/107 - 0028.jpg
처리 중: 6/107 - 0008.jpg
처리 중: 7/107 - 0041.jpg
처리 중: 8/107 - 0051.jpg
처리 중: 9/107 - 0059.jpg
처리 중: 10/107 - 0087.jpg
처리 중: 11/107 - 0100.jpg
처리 중: 12/107 - 0083.jpg
처리 중: 13/107 - 0021.jpg
처리 중: 14/107 - 0081.jpg
처리 중: 15/107 - 0068.jpg
처리 중: 16/107 - 0045.jpg
처리 중: 17/107 - 0017.jpg
처리 중: 18/107 - 0071.jpg
처리 중: 19/107 - 0016.jpg
처리 중: 20/107 - 0095.jpg
처리 중: 21/107 - 0060.jpg
처리 중: 22/107 - 0094.jpg
처리 중: 23/107 - 0103.jpg
처리 중: 24/107 - 0002.jpg
처리 중: 25/107 - 0012.jpg
처리 중: 26/107 - 0097.jpg
처리 중: 27/107 - 0065.jpg
처리 중: 28/107 - 0011.jpg
처리 중: 29/107 - 0007.jpg
처리 중: 30/107 - 0025.jpg
처리 중: 31/107 - 0056.jpg
처리 중: 32/107 - 0073.jpg
처리 중: 33/107 - 0082.jpg
처리 중: 34/107 - 0062.jpg
처리 중: 35/107 - 0023.jpg
처리 중: 36/107 - 0058.jpg
처리 중: 37/107 - 0032.jpg
처리 중: 38/107 - 0001.jpg
처리 중: 39/107 - 0026.jpg
처리 중: 40/107 - 0005.jpg
처리 중: 41/107 - 0019.jpg
처리 중: 42/107 - 0063.jpg
처

## 4. 결과 미리보기

In [108]:
# 첫 5개 결과 확인
for i, result in enumerate(results[:5]):
    print(f"\n이미지 {i+1}: {os.path.basename(result['image_path'])}")
    for key, value in result.items():
        if key != 'image_path' and not key.endswith('_prob'):
            prob = result.get(f"{key}_prob", 0.0)
            status = "✓" if value else "✗"
            print(f"  {status} {key}: {value} (확률: {prob:.3f})")


이미지 1: 0046.jpg
  ✗ has_step: False (확률: 0.312)
  ✓ has_movable_chair: True (확률: 0.970)
  ✗ has_high_chair: False (확률: 0.261)
  ✗ has_fixed_chair: False (확률: 0.507)
  ✗ has_floor_chair: False (확률: 0.290)

이미지 2: 0069.jpg
  ✗ has_step: False (확률: 0.443)
  ✓ has_movable_chair: True (확률: 0.941)
  ✗ has_high_chair: False (확률: 0.227)
  ✓ has_fixed_chair: True (확률: 0.840)
  ✗ has_floor_chair: False (확률: 0.129)

이미지 3: 0052.jpg
  ✗ has_step: False (확률: 0.256)
  ✓ has_movable_chair: True (확률: 0.974)
  ✗ has_high_chair: False (확률: 0.417)
  ✓ has_fixed_chair: True (확률: 0.820)
  ✗ has_floor_chair: False (확률: 0.189)

이미지 4: 0020.jpg
  ✗ has_step: False (확률: 0.389)
  ✗ has_movable_chair: False (확률: 0.432)
  ✗ has_high_chair: False (확률: 0.157)
  ✗ has_fixed_chair: False (확률: 0.611)
  ✓ has_floor_chair: True (확률: 0.771)

이미지 5: 0028.jpg
  ✗ has_step: False (확률: 0.444)
  ✓ has_movable_chair: True (확률: 0.957)
  ✗ has_high_chair: False (확률: 0.325)
  ✗ has_fixed_chair: False (확률: 0.428)
  ✗ has_floor_ch

## 5. CSV로 저장

In [109]:
# DataFrame으로 변환
df = pd.DataFrame(results)

# 파일명만 추출 (전체 경로 대신)
df['file_path'] = df['image_path'].apply(os.path.basename)

# image_path 열 제거 (file_path로 대체)
df = df.drop('image_path', axis=1)

# 열 순서 재정렬: file_path를 맨 앞으로
cols = ['file_path'] + [col for col in df.columns if col != 'file_path']
df = df[cols]

# 결과 저장 경로
OUTPUT_DIR = "../outputs/predictions"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 모델 타입 자동 감지 (SigLIP vs SigLIP2)
if 'siglip2' in MODEL_ID.lower():
    model_type = 'siglip2'
else:
    model_type = 'siglip'

# 파일명 생성 (모델 타입 + head type + fine-tuned/base)
if CHECKPOINT_PATH:
    # ML Decoder 감지
    if 'mldecoder' in CHECKPOINT_PATH.lower():
        model_name = f"{model_type}_mldecoder_ft"
    else:
        model_name = f"{model_type}_ft"  # fine-tuned linear head
else:
    model_name = f"{model_type}_base"  # base model

SAVE_PATH = os.path.join(OUTPUT_DIR, f'{model_name}_{THRESHOLD}.csv')

# CSV 저장
df.to_csv(SAVE_PATH, index=False, encoding='utf-8-sig')

print(f"\n결과가 저장되었습니다: {SAVE_PATH}")
print(f"총 {len(df)}개 이미지 분류 완료")
print(f"사용된 모델: {model_name} (threshold={THRESHOLD})")


결과가 저장되었습니다: ../outputs/predictions/siglip2_mldecoder_ft_0.75.csv
총 107개 이미지 분류 완료
사용된 모델: siglip2_mldecoder_ft (threshold=0.75)


## 6. 통계 확인

In [110]:
# 레이블별 감지 횟수
label_cols = ['has_step', 'has_movable_chair', 'has_high_chair', 'has_fixed_chair', 'has_floor_chair']

print("\n=== 레이블별 감지 통계 ===")
for label in label_cols:
    count = df[label].sum()
    ratio = count / len(df) * 100
    print(f"{label}: {count}/{len(df)} ({ratio:.1f}%)")


=== 레이블별 감지 통계 ===
has_step: 0/107 (0.0%)
has_movable_chair: 72/107 (67.3%)
has_high_chair: 7/107 (6.5%)
has_fixed_chair: 22/107 (20.6%)
has_floor_chair: 2/107 (1.9%)
